In [6]:
import torch
import torch.nn as nn
import einops
import tiktoken
from torch.nn import functional as F
from dataclasses import dataclass

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
print(device)

tokenizer = tiktoken.get_encoding('gpt2')

cuda


main changes:
* dataloader class
* wte = proj
* initialization stddev
* residual $\frac{1}{\sqrt{N}}$ in attention
* "nice" powers of 2 numbers

In [172]:
@dataclass
class GPTConfig:
    batch_size: int = 32
    block_size: int = 256
    vocab_size: int = 50304 # tokenizer.n_vocab # 50257, 50000 merges + 256 byte + <endoftext>
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    lr: float = 3e-4

In [ ]:
class DataLoader():
    def __init__(self, B, T):
        self.B = B
        self.T = T

        with open('../input.txt', 'r', encoding='utf-8') as f:
            text = f.read()

        data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
        n = int(0.9*len(data)) # first 90% will be train, rest val
        self.train_data = data[:n]
        self.val_data = data[n:]

        self.current_pos = 0
        print(f"tokens: {len(self.train_data)}")
        print(f"1 epoch is {len(self.train_data) // (B * T)} batches")

    def get_batch(self, split):
        batch_size, block_size = self.B, self.T
        data = self.train_data if split == 'train' else self.val_data
        ix = torch.randint(low=0, high=len(data)-block_size, size=(batch_size, ))

        x = torch.stack([data[i:i+block_size] for i in ix])
        y = torch.stack([data[i+1:i+block_size+1] for i in ix])
        x, y = x.to(device), y.to(device)
        return x, y
    
    def next_batch(self, split):
        B, T, = self.B, self.T
        data = self.train_data if split == 'train' else self.val_data
        buffer = data[self.current_pos : self.current_pos + B*T + 1]
        
        x = buffer[:-1].view(B, T)
        y = buffer[1:].view(B, T)
        x, y = x.to(device), y.to(device)

        self.current_pos += B*T
        if self.current_pos >= (len(data) - (1+B*T)):
            self.current_pos = 0
        return x, y

In [174]:
class FFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(approximate='tanh'),
            nn.Linear(4 * config.n_embd, config.n_embd)
        )
    
    def forward(self, x):
        return x + self.net(x)

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.ln1 = nn.LayerNorm(config.n_embd)
        self.multiattn = nn.MultiheadAttention(embed_dim=config.n_embd, num_heads=config.n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ffn = FFN(config)
    
    def forward(self, x):
        norm = self.ln1(x)
        attn, _ = self.multiattn(query=norm, key=norm, value=norm, need_weights=False, attn_mask=torch.triu(torch.ones(x.shape[-2], x.shape[-2], device=device, dtype=torch.bool), diagonal=1))
        x = x + attn
        x = x + self.ffn(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln = nn.LayerNorm(config.n_embd)
        ))

        self.proj = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.proj.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size, f"context length of {T} exceeds block_size of {self.config.block_size}"
        
        pos_emb = self.transformer.wpe(torch.arange(T, dtype=torch.long, device=device)) # (block, embed)
        tok_emb = self.transformer.wte(idx) # (B,T,C) -> (batch, block, embed)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln(x)

        if targets is None:
            loss = None
            logits = x[:, [-1], :] # only need to calc last ones for generation
            logits = self.proj(logits)
        else:
            logits = self.proj(x)
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)    
    
        return logits, loss


    def generate(self, idx, max_tokens=1, temp=1):
        for _ in range(max_tokens):
            logits, _ = self(idx[:,-self.config.block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs / temp, num_samples=1)
            idx = torch.cat((idx, ix), dim=1) # lol no need to cut
            # print(decode(ix[0].tolist()), end="", flush=True)
        return idx

In [175]:
loader = DataLoader(B=4, T=GPTConfig().block_size) # GPTConfig.block_size
loader.next_batch('train')

tokens: 304222
1 epoch is 297 batches


(tensor([[ 5962, 22307,    25,  ...,   198,   454,   279],
         [ 7938,    11,   304,  ...,    11,   351, 18201],
         [   11,   284, 15867,  ...,  1337,   198, 11980],
         [  262,  1458,  1173,  ...,  1842,   484,  6842]], device='cuda:0'),
 tensor([[22307,    25,   198,  ...,   454,   279,  7938],
         [   11,   304,   260,  ...,   351, 18201,    11],
         [  284, 15867,   287,  ...,   198, 11980,   262],
         [ 1458,  1173,  1547,  ...,   484,  6842,   514]], device='cuda:0'))

optimizations:
* BF16 and matmul precision `'high'`
* `torch.compile()`

In [176]:
model = GPT(GPTConfig())
model = model.to(device=device)

if torch.cuda.is_available() and torch.cuda.get_device_capability(device)[0] >= 7:
    model = torch.compile(model)
else:
    print("Triton only supports devices of CUDA Capability >= 7.0")

optimizer = torch.optim.AdamW(model.parameters(), lr=model.config.lr)

Triton only supports devices of CUDA Capability >= 7.0


In [12]:
import time

In [147]:
t0 = time.time()
print("hi")
t1 = time.time()
t1-t0

hi


0.001001596450805664

In [152]:
torch.set_float32_matmul_precision('high')

In [ ]:
for i in range(30): 
    t0 = time.time()

    Xb, Yb = loader.next_batch('train')
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device, dtype=torch.bfloat16): # slower on pascal arch
        logits, loss = model(Xb, Yb)

    loss.backward()
    norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    torch.cuda.synchronize()
    t1 = time.time()
    if i % 10 == 0:
        dt = (t1 - t0) 
        tokens = (loader.B * loader.T) / dt
        print(f"step: {i}, loss: {loss.item():.4f}, norm: {norm:.2f}, time: {1000 * dt:.2f} ms, tok/sec: {tokens:.2f}")
        t0 = time.time()

step: 0, loss: 6.5809, norm: 1.54, time: 545.13 ms, tok/sec: 1878.46
step: 10, loss: 6.5939, norm: 0.82, time: 169.75 ms, tok/sec: 6032.29
step: 20, loss: 6.8160, norm: 0.94, time: 174.52 ms, tok/sec: 5867.43
